# 01 — Подготовка аудио RuASD (CPU)
Блокнот читает зафиксированный CSV-манифест, показывает примеры, а затем по явному флагу готовит mono FLAC 32 кГц длиной 4 секунды. По умолчанию массовая обработка **не запускается**. Состояние хранится на Google Drive, поэтому ту же ячейку можно безопасно запустить после переподключения Colab.

In [ ]:
# 1. Среда и репозиторий
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Unvand/audio-deepfake-detector.git'
REPO_DIR = Path('/content/audio-deepfake-detector')
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab-prepare.txt')], check=True)
    os.chdir(REPO_DIR)
print('Python:', sys.version)
print('Рабочий каталог:', Path.cwd())

In [ ]:
# 2. Google Drive, HF_TOKEN и CSV-манифест
import pandas as pd
from IPython.display import Audio, display

if IN_COLAB:
    from google.colab import drive, files, userdata
    drive.mount('/content/drive')
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    HF_TOKEN = os.environ.get('HF_TOKEN')

MANIFEST_PATH = Path('/content/ruasd_final_manifest.csv')
if not MANIFEST_PATH.exists() and IN_COLAB:
    print('Загрузите приложенный ruasd_final_manifest.csv')
    uploaded = files.upload()
    if 'ruasd_final_manifest.csv' not in uploaded:
        raise FileNotFoundError('Нужен файл ruasd_final_manifest.csv')
if not HF_TOKEN:
    raise RuntimeError('Добавьте HF_TOKEN в Colab Secrets; токен не записывается в notebook')
manifest = pd.read_csv(MANIFEST_PATH)
print('Манифест:', MANIFEST_PATH, manifest.shape)

In [ ]:
# 3. Проверка готового split — новый случайный split не создаётся
from audio_deepfake_detector.src.ruasd_preparation import (
    PreparationConfig,
    RuASDShardReader,
    disk_space_report,
    estimate_prepared_size,
    prepare_ruasd_dataset,
    resample_mono,
    select_preview_examples,
    validate_ruasd_manifest,
    verify_prepared_dataset,
)

validation = validate_ruasd_manifest(manifest)
display(pd.crosstab(manifest['split'], manifest['label'], margins=True))
print(validation)
assert len(manifest) == 12_806
assert validation['unknown_duration'] == 321

## Прослушивание 12 train-примеров
Следующая ячейка читает каждый нужный shard один раз. `test_unseen` по умолчанию не используется. Для каждого примера доступны исходная и ресемплированная версии.

In [ ]:
# 4. Предпросмотр REAL/FAKE (сеть, CPU)
INCLUDE_UNSEEN_PREVIEW = False
preview_rows = select_preview_examples(manifest, include_unseen=INCLUDE_UNSEEN_PREVIEW)
reader = RuASDShardReader(HF_TOKEN, PreparationConfig())
for row, decoded, error in reader.iter_decoded(preview_rows):
    if error:
        print('ОШИБКА:', row['sample_id'], error)
        continue
    resampled = resample_mono(decoded.waveform, decoded.sample_rate, 32_000)
    print({
        'label': str(row['label']).upper(),
        'subset': row['subset'],
        'sample_id': row['sample_id'],
        'original_sample_rate': decoded.sample_rate,
        'actual_duration': decoded.waveform.size / decoded.sample_rate,
    })
    print('Исходное аудио:')
    display(Audio(decoded.waveform, rate=decoded.sample_rate))
    print('После ресемплинга до 32 кГц:')
    display(Audio(resampled, rate=32_000))

In [ ]:
# 5. Оценка объёма и свободного места (без загрузки аудио)
OUTPUT_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/prepared_v1')
CONFIG = PreparationConfig(target_sample_rate=32_000, segment_seconds=4.0, output_format='FLAC')
estimate = estimate_prepared_size(manifest, CONFIG)
space = disk_space_report(OUTPUT_ROOT, estimate['rough_flac_bytes_known'])
display(pd.Series(estimate, name='estimate').to_frame())
display(pd.Series(space, name='disk').to_frame())
if not space['enough_space']:
    print('ПРЕДУПРЕЖДЕНИЕ: оценки места с резервом 2 ГиБ недостаточно. Не запускайте FULL MODE.')
print('321 неизвестная длительность не включена в оценку, но будет измерена после декодирования.')

In [ ]:
# 6. TEST MODE: явно переключите флаг; повторный запуск проверяет resume
RUN_TEST_MODE = False
if RUN_TEST_MODE:
    test_summary = prepare_ruasd_dataset(
        manifest, OUTPUT_ROOT, HF_TOKEN, CONFIG,
        full_mode=False, test_limit=8, checkpoint_every=1,
    )
    print(test_summary)
    print(verify_prepared_dataset(OUTPUT_ROOT))
else:
    print('TEST MODE не запущен. Установите RUN_TEST_MODE = True.')

In [ ]:
# 7. FULL MODE: две независимые защиты от случайного запуска
RUN_FULL_MODE = False
FULL_MODE_CONFIRMATION = ''  # впишите: PREPARE ALL 12806 FILES
if RUN_FULL_MODE:
    if FULL_MODE_CONFIRMATION != 'PREPARE ALL 12806 FILES':
        raise RuntimeError('FULL MODE не подтверждён точной контрольной фразой')
    if not space['enough_space']:
        raise RuntimeError('Недостаточно места с учётом резерва')
    full_summary = prepare_ruasd_dataset(
        manifest, OUTPUT_ROOT, HF_TOKEN, CONFIG,
        full_mode=True, checkpoint_every=10,
    )
    print(full_summary)
else:
    print('FULL MODE не запущен.')

## Возобновление после отключения
Снова выполните ячейки 1–3 и 5, затем ту же ячейку TEST/FULL MODE с прежними `OUTPUT_ROOT` и `CONFIG`. Успешные исходные записи с существующими корректными файлами будут пропущены; ошибки и неполные записи — обработаны заново. Не меняйте конфигурацию в том же `prepared_v1`.